# Scattering kernels — graphite

**NJOY**

Compare the reference and three stochastic vibrational spectra using kernels that describe outgoing neutron energy at fixed incident energy. TARA evaluates the outgoing-energy kernel from the thermal scattering law in the NJOY MF7 output.

**NCrystal**

Compare the reference and three stochastic vibrational spectra using kernels that describe outgoing neutron energy at fixed incident energy. TARA evaluates the outgoing-energy kernel from the NCrystal graphite scattering law.

## 0. Choose the engine

**Required setting**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ENGINE` | `"ncrystal"` | Select `"njoy"` or `"ncrystal"` for this run. |

After changing the engine, restart the kernel and run all cells.

In [ ]:
AVAILABLE_ENGINE = ("njoy", "ncrystal")
AVAILABLE_ENGINE_LABEL = {
    "njoy": "NJOY",
    "ncrystal": "NCrystal"
}

In [ ]:
ENGINE = "ncrystal"
if ENGINE not in AVAILABLE_ENGINE:
    raise ValueError('Choose ENGINE = "njoy" or "ncrystal".')
ENGINE_LABEL = AVAILABLE_ENGINE_LABEL[ENGINE]

## 1. Locate TARA

The next cell finds TARA and imports the APIs for the chosen engine.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "core" / "config.py").is_file()
     and (path / "processing" / "pipelines" / "ace_pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside the TARA repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import parallel_execution
from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler

if ENGINE == "njoy":
    from input_output.leapr.core import LeaprFile
    from input_output.leapr.temperature import resolve_leapr_temperature
elif ENGINE == "ncrystal":
    from input_output.ncrystal.core import NCrystalFile
    import tempfile
    import NCrystal as NC

## 2. Select the required inputs

| Setting | Example value | What it controls |
| --- | --- | --- |
| `BASE_LEAPR_FILE` | `tests/fixtures/input_2.leapr` | NJOY material template. |
| `BASE_NCMAT_FILE` | `tests/fixtures/C_sg194_pyrolytic_graphite.ncmat` | NCrystal graphite template. |
| `BASE_ENDF_FILE` | `tests/fixtures/n_6-C-012g.jeff` | Neutron evaluation for the NJOY path. Override with `TARA_BASE_ENDF` if needed. |
| `NJOY_EXECUTABLE` | `NJOY_EXEC` or `njoy` on `PATH` | Executable for the NJOY path. |

Edit paths only if your checkout or NJOY installation uses different locations.

In [ ]:
fixtures = PROJECT_ROOT / "tests" / "fixtures"
BASE_ENDF_FILE = Path(os.environ.get("TARA_BASE_ENDF", str(fixtures / "n_6-C-012g.jeff")))
njoy_path = shutil.which(os.environ.get("NJOY_EXEC") or "njoy")
NJOY_EXECUTABLE = Path(njoy_path) if njoy_path else None

if ENGINE == "njoy":
    if not BASE_ENDF_FILE.is_file() or BASE_ENDF_FILE.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty neutron ENDF: {BASE_ENDF_FILE}")
    if NJOY_EXECUTABLE is None:
        raise FileNotFoundError("Set NJOY_EXEC or add njoy to PATH.")

    print("Neutron ENDF:", BASE_ENDF_FILE)
    print("NJOY:", NJOY_EXECUTABLE)

In [ ]:
if ENGINE == "njoy":
    BASE_LEAPR_FILE = fixtures / "input_2.leapr"
    if not BASE_LEAPR_FILE.is_file():
        raise FileNotFoundError(f"LEAPR template missing: {BASE_LEAPR_FILE}")

elif ENGINE == "ncrystal":
    BASE_NCMAT_FILE = fixtures / "C_sg194_pyrolytic_graphite.ncmat"
    if not BASE_NCMAT_FILE.is_file():
        raise FileNotFoundError(f"NCMAT template missing: {BASE_NCMAT_FILE}")

## 3. Load the graphite reference

**Fixed settings for this example**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `TEMPERATURE_K` | `293.6` | Evaluation temperature in kelvin. |
| `ATOM_LABEL` | `"C"` | Species supplying the reference spectrum. |

In [ ]:
TEMPERATURE_K = 293.6
ATOM_LABEL = "C"
print(f"Engine: {ENGINE_LABEL}")
print(f"temperature: {TEMPERATURE_K:g} K")

if ENGINE == "njoy":
    template = LeaprFile.from_file(str(BASE_LEAPR_FILE))
    leapr_template = template

    resolve_leapr_temperature(template, TEMPERATURE_K, warn_on_reuse=False)
    raw_pdos = template.pdos
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"LEAPR file: {BASE_LEAPR_FILE}")
elif ENGINE == "ncrystal":
    material = NCrystalFile.from_file(str(BASE_NCMAT_FILE))
    raw_pdos = material.pdos_for_atom(ATOM_LABEL)
    reference_spectrum = PhononSpectra(pdos=raw_pdos)
    print(f"NCMAT file: {BASE_NCMAT_FILE}")

## 4. Sample the vibrational spectrum

**Required:** The PDOS from Section 3 and the two perturbation bounds below. Select a positive sample count for the comparison.

**Required sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `1.0` | Maximum phonon-energy displacement in meV. |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative PDOS amplitude change in percent. |
| `N_SAMPLES` | `3` | Number of stochastic samples. The reference is stored separately. |

In [ ]:
MAX_ENERGY_SHIFT_MEV = 1.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3

**Optional sampling settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `20261008` | Seed for reproducible sampling. |
| `BATCH_SIZE` | `512` | Internal sampling chunk size. |
| `AMPLITUDE_METHOD` | `"fast"` | Method used for the amplitude perturbations. |
| `SPATIAL_METHOD` | `"diffeomorphic"` | Method used to shift the spectrum in energy. |

In [ ]:
RANDOM_SEED = 20261008
BATCH_SIZE = 512
AMPLITUDE_METHOD = "fast"
SPATIAL_METHOD = "diffeomorphic"

In [ ]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
)

# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum,
    amplitude_method=AMPLITUDE_METHOD,
    spatial_method=SPATIAL_METHOD,
)

# Generate the stochastic ensemble with the configured sampling batch size.
ensemble = sampler.generate_ensemble(batch_size=BATCH_SIZE)
if ensemble.reference is None or len(ensemble) != N_SAMPLES:
    raise RuntimeError("Expected the reference and all requested samples.")
print(f"Prepared one reference and {len(ensemble)} stochastic spectra.")

## 5. Configure the Analyzer and request scattering kernels

**Required:** The material from Section 3 and the ensemble from Section 4. Use the backend runner and parser for the material selected above.

In [ ]:
import numpy as np
from analyzer.ensemble_analyzer import EnsembleAnalyzer

if ENGINE == "njoy":
    from endf_parserpy import EndfParserPy
    from scipy.constants import physical_constants
    from analyzer.parsers.njoy_parser import NjoyParser
    from analyzer.runners.njoy_runner import NjoyRunner

    base_header = EndfParserPy().parsefile(str(BASE_ENDF_FILE), include=[1])[1][451]
    if int(round(float(base_header["ZA"]))) != 6012:
        raise ValueError("Expected base neutron evaluation ZA=6012.")
    MASS_AMU = float(template.principal_scatterer["awr"]) * physical_constants["neutron mass in u"][0]
elif ENGINE == "ncrystal":
    from analyzer.parsers.ncrystal_parser import NCrystalParser
    from analyzer.runners.ncrystal_runner import NCrystalRunner

In [ ]:
# Configure the appropriate engine runner and parser based on the selected ENGINE.
if ENGINE == "njoy":
    runner = NjoyRunner(
        njoy_exec=str(NJOY_EXECUTABLE),
        base_endf_file=str(BASE_ENDF_FILE),
        base_leapr_file=str(BASE_LEAPR_FILE),
        temperature=TEMPERATURE_K,
        mass_amu=MASS_AMU,
    )
    parser = NjoyParser()
elif ENGINE == "ncrystal":
    runner = NCrystalRunner(
        base_ncmat_file=str(BASE_NCMAT_FILE),
        temperature=TEMPERATURE_K,
    )
    parser = NCrystalParser()

In [ ]:
# Initialize the ensemble analyzer with the generated ensemble, runner, and parser.
analyzer = EnsembleAnalyzer(
    ensemble=ensemble,
    runner=runner,
    parser=parser
)

**Required observable settings**

| Setting | Example | What it controls |
| --- | --- | --- |
| `INCIDENT_ENERGIES` | `[0.0253, 0.05, 0.1, 0.5, 1.0]` eV | Incident energies evaluated for the reference and samples. |

**Optional observable setting**

| Setting | Example | What it controls |
| --- | --- | --- |
| `THETA_GRID` | 0°–180° in 1° steps | Angular quadrature for the integral over scattering cosine. Keep both endpoints when changing the resolution. |

The Analyzer integrates the differential scattering response over scattering cosine to obtain each outgoing-energy kernel. It stores a series indexed by outgoing energy in eV for each incident energy. The series retains the Analyzer’s native normalization.

In [ ]:
INCIDENT_ENERGIES = np.array([0.0253, 0.05, 0.1, 0.5, 1.0], dtype=float)
THETA_GRID = np.arange(181, dtype=float)

parser_kwargs = dict(
    calc_scattering_kernel=True,
    ein_arr=INCIDENT_ENERGIES,
    theta=THETA_GRID,
    calc_xs_total=False,
    calc_xs_inelastic=False,
    calc_xs_elastic=False,
    calc_xs_absorption=False,
)

## 6. Run the Analyzer

**NJOY**

**Required:** The Analyzer and observable settings from Section 5. Evaluate the reference separately so Section 7 can overlay it with the stochastic kernels.

**Optional execution and THERMR settings:**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RUN_PARALLEL` | `False` | Run sequentially or request concurrent processing. |
| `thermr_emax` | `5.0` | Upper THERMR processing energy in eV. |
| `thermr_tol` | `0.05` | THERMR interpolation tolerance. |
| `thermr_nbin` | `16` | THERMR angular bin count. |

**NCrystal**

**Required:** The Analyzer and observable settings from Section 5 and the atom label from Section 3. Evaluate the reference separately so Section 7 can overlay it with the stochastic kernels.

**Optional setting:** `RUN_PARALLEL` controls concurrent execution.

In [ ]:
RUN_PARALLEL = False

if ENGINE == "njoy":
    RUNNER_OPTIONS = {
        "thermr_emax": 5.0,
        "thermr_tol": 0.05,
        "thermr_nbin": 16,
    }
elif ENGINE == "ncrystal":
    RUNNER_OPTIONS = {
        "atom_label": ATOM_LABEL
    }

In [ ]:
# Compute the ensemble results using the configured analyzer.
analyzer.compute(
    parallel=RUN_PARALLEL,
    runner_kwargs=RUNNER_OPTIONS,
    parser_kwargs=parser_kwargs,
    compute_reference=True,
)

if analyzer.reference_result is None or len(analyzer.raw_results) != len(ensemble):
    raise RuntimeError("The Analyzer did not return the expected reference and sample results.")
print(f"Analyzed one reference and {len(analyzer.raw_results)} stochastic spectra.")

## 7. Plot scattering kernels

**Required:** The reference and sample results from Section 6. Select an incident energy to compare the kernels versus outgoing neutron energy.

**Optional display setting:** `KERNEL_RELATIVE_THRESHOLD` controls the outgoing-energy range shown. The slider reads the stored results. If `ipywidgets` is unavailable, the notebook renders the first incident energy.

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import display
from notebooks.tara_plotting import plot_PDOS, plot_scattering_kernel

REF_LABEL = f"{ENGINE_LABEL} reference"
KERNEL_RELATIVE_THRESHOLD = 1e-4
incident_energies = tuple(float(energy) for energy in INCIDENT_ENERGIES)

def kernel_figure(incident_energy):
    """Show graphite VDOS and the selected scattering kernel."""
    fig, (pdos_axis, result_axis) = plt.subplots(
        2, 1, figsize=(12, 8), layout="constrained",
        gridspec_kw={"height_ratios": (1, 1.3)},
    )
    plot_PDOS(pdos_axis, list(ensemble), ensemble.reference,
              title="Reference and stochastic VDOS", ref_label=REF_LABEL)
    plot_scattering_kernel(
        result_axis, analyzer, incident_energy, ref_label=REF_LABEL,
        relative_threshold=KERNEL_RELATIVE_THRESHOLD,
    )
    return fig

try:
    import ipywidgets as widgets
except ImportError:
    figure = kernel_figure(incident_energies[0])
    display(figure)
    plt.close(figure)
else:
    energy_selector = widgets.SelectionSlider(
        options=[(f"{energy:g} eV", energy) for energy in incident_energies],
        description="Incident energy",
        continuous_update=False,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="min(650px, 100%)"),
    )
    plot_image = widgets.Image(format="png", layout=widgets.Layout(width="100%"))

    def update_kernel(change=None):
        """Redraw the selected kernel using stored Analyzer results."""
        figure = kernel_figure(energy_selector.value)
        try:
            with io.BytesIO() as buffer:
                figure.savefig(buffer, format="png", dpi=120)
                plot_image.value = buffer.getvalue()
        finally:
            plt.close(figure)

    energy_selector.observe(update_kernel, names="value")
    update_kernel()
    display(widgets.VBox([energy_selector, plot_image]))